# Helpers to work with files of data

## Setup
imdb data set as an example

In [3]:
from pathlib import Path
import tensorflow as tf

root = "https://ai.stanford.edu/~amaas/data/sentiment/"
filename = "aclImdb_v1.tar.gz"
filepath = tf.keras.utils.get_file(filename, root + filename, extract=True,
                                   cache_dir=".")
if "_extracted" in filepath:
    path = Path(filepath) / "aclImdb"
else:
    path = Path(filepath).with_name("aclImdb")


In [4]:
# define paths
train_pos_dr = path / "train" / "pos"
train_neg_dr = path / "train" / "neg"
test_pos_dr = path / "test" / "pos"
test_neg_dr = path / "test" / "neg"

## Parse files with Dataset

List files as Path objects:

In [5]:
from pathlib import Path
import numpy as np

def list_txt(dirpath: Path) -> list[Path]:
    return list(dirpath.glob("*.txt"))

# list files as Path objects
train_pos_files = list_txt(train_pos_dr)
train_neg_files = list_txt(train_neg_dr)
test_pos_files  = list_txt(test_pos_dr)
test_neg_files  = list_txt(test_neg_dr)


In [6]:
from numpy.random import shuffle
from tensorflow.data import Dataset
from typing import Sequence

def make_text_ds(filepaths: Sequence[Path]) -> Dataset:
  paths_as_str = [str(p) for p in filepaths]
  ds = Dataset.from_tensor_slices(paths_as_str)
  ds = ds.map(tf.io.read_file, num_parallel_calls = tf.data.AUTOTUNE)
  return ds

def make_labeled_ds(
    pos_files: Sequence[str],
    neg_files: Sequence[str],
    shuffle=True,
    cache=False):

  pos_ds = make_text_ds(pos_files).map(lambda x: (x, 1))
  neg_ds = make_text_ds(neg_files).map(lambda x: (x, 0))
  ds = pos_ds.concatenate(neg_ds)

  if shuffle:
    ds = ds.shuffle(len(ds))

  if cache:
    ds = ds.cache()

  return ds

train_ds = make_labeled_ds(train_pos_files, train_neg_files, shuffle=True)
test_ds = make_labeled_ds(test_pos_files, test_neg_files, shuffle=False)